# B2 · Tabloya yeni sütun eklemek

**Bugünün tek fikri:** var olan sütunlardan **yeni bir sütun türetmek**: `tablo["yeni"] = <ifade>`. Cuma günkü B3'te (groupby) gruplayacağın sütunları (mevsim, saat, rüzgâr sınıfı) bugün kendin üreteceksin.

**Kural (B1'deki gibi):** her kod hücresinden önce altındaki kutuya "bu ne yapıyor" diye tek cümle yaz. Sonra çalıştırıp kontrol et.

## 1. Oyuncak tablo

Gene 5 satır. `olcum` ve `gfs` knot cinsinden, `saat_utc` 0–23 arası saat, `ay` 1–12 arası.

In [ ]:
import pandas as pd

oyuncak = pd.DataFrame({
    "olcum":    [4, 12, 18, 22, 9],
    "gfs":      [5, 8, 10, 21, 9],
    "saat_utc": [2, 9, 12, 22, 23],
    "ay":       [1, 4, 7, 7, 11],
})
oyuncak

✍️ *Bu ne yapıyor? (tek cümle)*



### 1a. İki sütundan yeni sütun

`oyuncak["olcum"] - oyuncak["gfs"]` her satırda **kendi** değerleri birbirinden çıkarır. Döngü yazmana gerek yok. Sonucu `oyuncak["fark"] = ...` ile tabloya yeni bir sütun olarak yazıyorsun.

In [ ]:
oyuncak["fark"] = oyuncak["gfs"] - oyuncak["olcum"]
oyuncak

✍️ *Bu ne yapıyor? (tek cümle)*



### 1b. Maske de bir sütun olabilir

B1'deki maske (True/False Series) aynen tabloya yazılabiliyor. İleride "kuvvetli saatler" diye sürekli yeniden hesaplamak yerine sütundan okursun.

In [ ]:
oyuncak["kuvvetli"] = oyuncak["olcum"] >= 15
oyuncak

✍️ *Bu ne yapıyor? (tek cümle)*



### 1c. Sütun üstünde hesap: `+`, `%`, `.abs()`

UTC'yi İzmir saatine çevirmek: +3 ekle, 24'e göre kalanını al (`%`). 22+3=25 olunca 1'e döner.

In [ ]:
oyuncak["yerel_saat"] = (oyuncak["saat_utc"] + 3) % 24
oyuncak["mutlak_fark"] = oyuncak["fark"].abs()
oyuncak

✍️ *Bu ne yapıyor? (tek cümle)*



### 1d. Sözlükle çevirmek: `.map()`

Her değeri bir sözlükten karşılığıyla değiştirir. `ay` sütunundaki 7, sözlükteki `7: "yaz"` ile `"yaz"` olur.

In [ ]:
mevsim_sozluk = {12: "kis", 1: "kis", 2: "kis",
                 3: "ilkbahar", 4: "ilkbahar", 5: "ilkbahar",
                 6: "yaz", 7: "yaz", 8: "yaz",
                 9: "sonbahar", 10: "sonbahar", 11: "sonbahar"}
oyuncak["mevsim"] = oyuncak["ay"].map(mevsim_sozluk)
oyuncak

✍️ *Bu ne yapıyor? (tek cümle)*



### 1e. Sayıyı aralığa çevirmek: `pd.cut`

`bins` sınırları verir. `right=False` olunca aralık `[10, 15)` olur, yani 10 dahil, 15 hariç. `labels` her aralığın adı.

In [ ]:
oyuncak["sinif"] = pd.cut(oyuncak["olcum"],
                         bins=[0, 10, 15, 20, 100],
                         labels=["hafif", "orta", "iyi", "kuvvetli"],
                         right=False)
oyuncak

✍️ *Bu ne yapıyor? (tek cümle)*



### 1f. Yeni sütuna bakmak: `.value_counts()` ve `.sort_values()`

`value_counts`: her değerden kaç tane var. `sort_values`: tabloyu bir sütuna göre sırala (`ascending=False` büyükten küçüğe).

In [ ]:
print(oyuncak["mevsim"].value_counts())
oyuncak.sort_values("mutlak_fark", ascending=False)

✍️ *Bu ne yapıyor? (tek cümle)*



## 2. Gerçek veri

Şimdi aynı şeyleri sen yazıyorsun. Her hücrede iskelet ve yorum satırı var. Takılırsan önce yukarıdaki oyuncak hücreye bak, sonra Pezo'ya ipucu sor.

In [ ]:
from ruzgar.data import load_aligned
veri = load_aligned()
veri.head()

### Alıştırma 1 · `hata` sütununu kendin üret

Tabloda zaten `hata` diye bir sütun var. İlk satıra bak: gfs 1,8, ölçüm 2,0, hata −0,2. **Önce** hangi çıkarma olduğunu tahmin et, sonra `benim_hata` sütununu yaz ve ikisinin aynı olduğunu kontrol et (en büyük fark ~0 çıkmalı).

In [ ]:
# veri["benim_hata"] = ...
# (veri["benim_hata"] - veri["hata"]).abs().max()


### Alıştırma 2 · Ham GFS'in ortalama mutlak hatası

`mutlak_hata` sütunu ekle (`.abs()`), sonra ortalamasını al. Bu sayı, rüzgâr projesindeki "ham GFS" MAE'sinin bütün veri üstündeki hali.

In [ ]:
# veri["mutlak_hata"] = ...
# ...mean()


### Alıştırma 3 · Yerel saat ve kontrol

`yerel_saat` sütunu ekle (`veri["time"].dt.hour` ile başla). Sonra `value_counts()` ile her saatten kaç tane olduğuna bak. Hepsi yaklaşık aynı mı? Neden öyle olmalı?

In [ ]:
# veri["yerel_saat"] = ...
# veri["yerel_saat"].value_counts().sort_index()


### Alıştırma 4 · Mevsim

`mevsim` sütununu ekle (`.dt.month` + yukarıdaki `mevsim_sozluk`). Her mevsimden kaç saat var? Veri Mart 2024'te başlıyor; sayılar eşit değilse nedeni bu olabilir mi?

In [ ]:
# veri["mevsim"] = ...
# ...value_counts()


### Alıştırma 5 · Rüzgâr sınıfı

`sinif` sütununu **ölçülen** hızdan `pd.cut` ile üret (aynı bins ve labels). Kaç saat "kuvvetli"? B1'deki 789 ile karşılaştır.

In [ ]:
# veri["sinif"] = pd.cut(...)
# ...value_counts()


### Alıştırma 6 · GFS'in en kötü 5 saati

`mutlak_hata`'ya göre büyükten küçüğe sırala, ilk 5 satırda yalnız `time`, `speed_kn_obs`, `speed_kn_gfs`, `hata` sütunlarını göster. Bu saatlerde GFS rüzgârı fazla mı tahmin etmiş, eksik mi?

In [ ]:
# veri.sort_values(...).head(5)[[...]]


## 3. Kapanış (2 cümle)

✍️ *Bugün öğrendiğim tek şey:*

✍️ *Cuma B3'te bu sütunlarla hangi soruyu sormak isterdim? (ör. "hangi mevsimde GFS en çok yanılıyor?")*
